# Telco Customer Churn — Data Wrangling Notebook (Outline Only)

**Author:** \<your name\>  
**Date:** 2025-08-27  
**Scope:** This notebook outlines the *data wrangling* steps for the Telco Customer Churn capstone. It includes checklists, rationale, and empty code cells for you to complete. No code is provided here by design.

> Dataset reference: *Telco Customer Churn* (Kaggle). Typical filename: `WA_Fn-UseC_-Telco-Customer-Churn.csv`.

## Objectives
- Produce a clean, analysis-ready dataset for EDA and modeling.
- Document assumptions and decisions so work is reproducible.
- Preserve an immutable copy of raw data.

## Success Criteria (Wrangling phase)
- ✅ All columns have correct dtypes (e.g., `TotalCharges` numeric, date-like fields parsed if present).
- ✅ Missing values are audited and addressed with transparent rules.
- ✅ Categorical values are standardized (consistent spelling/labels).
- ✅ Duplicates and impossible values are handled.
- ✅ Final tidy tables saved to `data/processed/` with a concise data dictionary.


## Wrangling Checklist

- [ ] Import libraries and set a deterministic environment (random seeds, display options).
- [ ] Load raw dataset(s) from `data/raw/`.
- [ ] Quick shape + peek (rows, columns, head/tail, sample).
- [ ] Column-by-column data audit (dtype, unique values, missingness, value ranges).
- [ ] Fix known Telco-specific issues (see below).
- [ ] Handle missing values (document strategy per column).
- [ ] Standardize/normalize categorical labels.
- [ ] Resolve duplicates and ID integrity.
- [ ] Detect and address outliers / impossible values (if applicable).
- [ ] Create derived helper columns needed for EDA (e.g., tenure buckets).
- [ ] Save cleaned dataset(s) to `data/processed/` and write a mini data dictionary.


## Telco Dataset — Common Wrangling Notes

- `customerID` should be unique. Verify no duplicates.
- `SeniorCitizen` is often encoded as `0/1` (numeric). Consider converting to a meaningful categorical label (`No`/`Yes`).
- `TotalCharges` may arrive as text with blanks/spaces for new customers. Treat blanks as missing and coerce to numeric.
- Contract and services columns (`Contract`, `InternetService`, `OnlineSecurity`, `TechSupport`, etc.) are categorical; standardize label capitalization and spelling.
- `Churn` is the target (Yes/No). Ensure it’s consistent and not inadvertently altered during cleaning.
- Consider engineering interpretable helper features for EDA (examples below), but keep the *modeling* feature engineering separate for traceability.


## 0. Environment & Imports (no code provided here)

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport

# Warning management
import warnings
from pandas.errors import SettingWithCopyWarning

# Suppress noisy warnings
warnings.filterwarnings(
    "ignore", category=UserWarning, module="ydata_profiling"
)  # profiling library benign messages

# Reproducibility seed (if needed)
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Display options (tweak as needed)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
# pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print("Environment ready | pandas", pd.__version__, "| numpy", np.__version__)

Environment ready | pandas 2.3.0 | numpy 2.1.3


## 1. Load Raw Data

**Goal:** Read the raw CSV(s) from `data/raw/` without altering them.  
**Deliverables:** In-memory DataFrame(s) for inspection; optional Parquet copy in `data/interim/` for speed.


In [2]:
telco_raw = pd.read_csv('/Users/Allison/Code/springboard_ds/springboard_projects/Capstone_2/data/raw/Telco-Customer-Churn-Blastchar.csv')

telco_raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Make a copy for wrangling
telco_wrangling = telco_raw.copy()


## 2. Initial Inspection

- Shape (rows × columns)
- First/last 5 rows and a random sample
- Column names (confirm expected schema)
- Basic info (`dtypes`, non-null counts) and simple `describe()` for numeric/categorical


In [19]:
print('HEAD:\n', telco_wrangling.head())
print('\nTAIL:\n', telco_wrangling.tail())
print('\nSAMPLE:\n', telco_wrangling.sample(n=5, random_state=SEED))
print('\nSHAPE:\n', telco_wrangling.shape)
print('\nINFO:')
telco_wrangling.info()
print('\nDESCRIBE:\n', telco_wrangling.describe().T)
print('\nNULL COUNTS:\n', telco_wrangling.isnull().sum().sort_values(ascending=False))

HEAD:
    customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity  \
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No   
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes                No             DSL            Yes   
3  7795-CFOCW    Male              0      No         No      45           No  No phone service             DSL            Yes   
4  9237-HQITU  Female              0      No         No       2          Yes                No     Fiber optic             No   

  OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  \
0          Yes               No          No          No              No  Month-to-m

## 3. Draft Data Dictionary (from raw)

Create a simple table capturing:
- Column name
- Meaning / description
- Type (categorical, numeric, boolean, identifier, target)
- Expected domain / allowable values
- Notes (e.g., known issues, transformations planned)


In [9]:
telco_data_dict_raw = pd.read_csv('/Users/Allison/Code/springboard_ds/springboard_projects/Capstone_2/references/telco_raw_data_dictionary.csv')

print(telco_data_dict_raw)

         Column Name Data Type  Unique Values                                    Expected Values  \
0         customerID    object           7043  ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' '7795-...   
1             gender    object              2                                  ['Female' 'Male']   
2      SeniorCitizen     int64              2                                              [0 1]   
3            Partner    object              2                                       ['Yes' 'No']   
4         Dependents    object              2                                       ['No' 'Yes']   
5             tenure     int64             73                                   [ 1 34  2 45  8]   
6       PhoneService    object              2                                       ['No' 'Yes']   
7      MultipleLines    object              3                    ['No phone service' 'No' 'Yes']   
8    InternetService    object              3                         ['DSL' 'Fiber optic' 'No']   


## 4. Data Quality Audit

### 4.1 Missingness
- Overall missingness % by column
- Row-wise missingness distribution
- Investigate patterns (e.g., specific services not applicable by contract type)

In [ ]:
# Check for missing values -- NaN or None
missing = pd.concat([telco_wrangling.isnull().sum(), 100 * telco_wrangling.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count', ascending=False)

,count,%
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


In [21]:
# Check for blank or whitespace-only strings in object columns
for col in telco_wrangling.select_dtypes(include='object'):
    n_blanks = (telco_wrangling[col].str.strip() == '').sum()
    if n_blanks > 0:
        print(f"{col}: {n_blanks} blank/whitespace values")

TotalCharges: 11 blank/whitespace values


In [ ]:
# Check for placeholder values
placeholders = ['NA', 'N/A', 'null', 'None', '?', '-']
for col in telco_wrangling.columns:
    for val in placeholders:
        n = (telco_wrangling[col] == val).sum()
        if n > 0:
            print(f"{col}: {n} occurrences of '{val}'")

In [ ]:
# Check for constant columns
for col in telco_wrangling.columns:
    if telco_wrangling[col].nunique(dropna=False) == 1:
        print(f"{col} has only one unique value: {telco_wrangling[col].unique()}")

### 4.2 Duplicates & Keys
- Check `customerID` uniqueness
- Assess for full-duplicate rows

In [ ]:
# number unique values
telco_wrangling.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

In [35]:
# View unique values per column
for col in telco_wrangling.columns:
    if telco_wrangling[col].nunique() < 5:
        print(telco_wrangling[col].value_counts(), "\n")

gender
Male      3555
Female    3488
Name: count, dtype: int64 

SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64 

Partner
No     3641
Yes    3402
Name: count, dtype: int64 

Dependents
No     4933
Yes    2110
Name: count, dtype: int64 

PhoneService
Yes    6361
No      682
Name: count, dtype: int64 

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64 

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64 

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64 

OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64 

DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64 

TechSupport
No                     3473
Yes                    2044
No internet ser

### 4.3 Data Types
- Identify columns that need coercion (e.g., `TotalCharges` → numeric)
- Date-like parsing (if any date fields exist in your variant)

In [ ]:
# coerce TotalCharges to numeric
telco_wrangling['TotalCharges'] = pd.to_numeric(telco_wrangling['TotalCharges'], errors='coerce')

In [37]:
telco_wrangling.select_dtypes('object')

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
0,7590-VHVEG,Female,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,No
1,5575-GNVDE,Male,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,No
2,3668-QPYBK,Male,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,Yes
3,7795-CFOCW,Male,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),No
4,9237-HQITU,Female,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,No
7039,2234-XADUH,Female,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),No
7040,4801-JZAZL,Female,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
7041,8361-LTMKD,Male,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,Yes


### 4.4 Value Ranges & Invariants
- Validate tenure ranges (non-negative, reasonable max)
- Validate billing amounts (non-negative, plausible)
- Check categorical domains against expected labels

## 5. Apply Telco-Specific Fixes

Examples to consider (document what you do and why):
- Coerce `TotalCharges` to numeric after treating blanks as missing.
- Convert `SeniorCitizen` 0/1 to `No`/`Yes` or keep as binary consistent with modeling plan.
- Normalize categorical label capitalization/spelling (e.g., `No internet service` → `No internet` if desired, but keep a mapping table for reproducibility).


## 6. Missing Values — Strategy & Implementation

For each affected column, specify:
- Business logic for missingness (e.g., N/A because service not subscribed vs. true missing)
- Chosen approach (impute, flag, drop, conditional rules)
- Rationale and expected impact


## 7. Categorical Labels — Standardization

- Create a mapping table for each categorical with inconsistent labels.
- Optionally collapse rare categories (document threshold and motivation).
- Ensure consistent data types and category orders where meaningful.


## 8. Outliers & Impossible Values

- Define column-specific plausibility checks.
- Decide on winsorization/capping/removal rules *only if justified*.
- Record flagged rows to `data/interim/` for traceability.


## 9. Helper Features for EDA (Optional, Non-Model)

Examples (ensure they are interpretable and reversible):
- `tenure_group` (e.g., 0–6, 7–12, 13–24, 25–48, 49+ months)
- Count of subscribed services
- Binary indicators for families of services (e.g., any add-on security/support)
- MonthlyCharge bins for early EDA visualization


## 10. Finalize Clean Dataset

- Re-run `info()` and missingness checks to confirm.
- Save cleaned table(s) to `data/processed/` (CSV and/or Parquet).
- Write/update a concise data dictionary for the *processed* dataset.
- Log all transformations in a short changelog section below.


## 11. Quality Gates (Sanity Checks)

- No duplicated `customerID`.
- No unexpected nulls in required fields (document exceptions).
- Numeric columns within plausible ranges.
- Target `Churn` intact with expected class distribution.


## 12. Handoff to EDA (Next Notebook)

Export lightweight artifacts for the EDA notebook:
- Clean dataset path(s)
- Column role manifest (ID, target, numeric, categorical, booleans)
- Notes on any data caveats to keep in mind during EDA


In [5]:
# Helper function to log data wrangling changes
import os
from datetime import datetime
from typing import List, Optional

CHANGELOG_PATH = os.path.join(
    os.path.dirname(__file__) if "__file__" in globals() else 
    "/Users/Allison/Code/springboard_ds/springboard_projects/Capstone_2/data/interim",
    "wrangling_changelog.csv"
)

# Ensure directory exists (adjust if path resolution differs in your environment)
os.makedirs(os.path.dirname(CHANGELOG_PATH), exist_ok=True)

import pandas as _pd

def log_change(change: str, rationale: str, columns: Optional[List[str]] = None, date: Optional[str] = None):
    """Append a wrangling changelog entry (CSV) and print a Markdown row.

    Parameters
    ----------
    change : str
        Short description of the transformation applied.
    rationale : str
        Why this was done (business / data quality justification).
    columns : list[str] | None
        List of affected column names. Use None or empty for many/all.
    date : str | None
        Override date (YYYY-MM-DD). Defaults to today.
    """
    date_str = date or datetime.utcnow().strftime("%Y-%m-%d")
    col_field = ";".join(columns) if columns else "All/Multiple"

    row = {
        "date": date_str,
        "change": change,
        "rationale": rationale,
        "affected_columns": col_field,
    }

    if os.path.exists(CHANGELOG_PATH):
        existing = _pd.read_csv(CHANGELOG_PATH)
        updated = _pd.concat([existing, _pd.DataFrame([row])], ignore_index=True)
    else:
        updated = _pd.DataFrame([row])
    updated.to_csv(CHANGELOG_PATH, index=False)

    # Emit markdown row for notebook appendix table
    md_row = f"| {date_str} | {change} | {rationale} | {col_field} |"
    print(md_row)
    return row

# Example (commented):
# log_change(
#     change="Coerced TotalCharges to numeric (blank -> NaN)",
#     rationale="Needed numeric type; blanks signify new customers",
#     columns=["TotalCharges"],
# )

### Helper: Changelog Logger

Use the function below to append a machine-readable changelog entry and generate a Markdown row to paste into Appendix A.

Call pattern:
```python
log_change(
    change="Coerced TotalCharges to numeric (blanks -> NaN)",
    rationale="Enable numeric operations; blanks indicate new customers",
    columns=["TotalCharges"]
)
```
This will:
1. Append (or create) `data/interim/wrangling_changelog.csv`.
2. Print a Markdown table row you can paste under Appendix A.


---
## Appendix A — Data Changelog (Fill as you go)

Record each change with a brief justification and timestamp.

| Date | Change | Rationale | Affected Columns |
|------|--------|-----------|------------------|
| 2025-08-28 | Loaded raw CSV and created working copy `telco_wrangling` from `telco_raw` | Preserve immutable raw; all transformations target the copy | All |

## Appendix B — Assumptions & Open Questions

- Assumption:
- Open question:
